In [1]:
import numpy as np
from scipy.sparse import hstack
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sentence_transformers import SentenceTransformer
from scipy.sparse import csr_matrix

c:\Users\prakh\projects\discusso-ml\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
## Let's build a helper function to make error_df
import pandas as pd

def build_error_df(df_test, y_true, y_pred, model_name, fold_number, test_idx):
    """
    Builds error dataframe with stable row identity.
    """

    out = df_test.copy()

    # CRITICAL: preserve original row position
    out["original_index"] = df_test.index
    out["row_id"] = test_idx

    out["true_label"] = y_true
    out["pred_label"] = y_pred
    out["correct"] = y_true == y_pred
    out["model"] = model_name
    out["fold"] = fold_number

    return out

In [18]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

def generate_stratified_folds(df, label_col="effort", n_splits=5):
    """
    Generates folds using stable row positions.
    """

    #use position indices (not df.index)
    X = np.arange(len(df))
    y = df[label_col].values

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    folds = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        folds.append({
            "fold": fold,
            "train_idx": train_idx,
            "test_idx": test_idx
        })

    return folds

In [19]:
import pandas as pd


def aggregate_fold_features(
    fold_features,
    label_key,
    top_k=10
):
    """
    Aggregates TF-IDF features across folds.

    Parameters
    ----------
    fold_features : list of dicts
        Each dict contains label_0_features / label_1_features DataFrames
    label_key : str
        Either "label_0_features" or "label_1_features"
    """

    rows = []

    for fold_data in fold_features:
        df = fold_data[label_key].copy()
        rows.append(df)

    all_features = pd.concat(rows)

    aggregated = (
        all_features
        .groupby("feature")
        .agg(
            freq=("feature", "count"),
            mean_coef=("coefficient", "mean"),
            mean_abs_coef=("coefficient", lambda x: x.abs().mean())
        )
        .sort_values(
            by=["freq", "mean_abs_coef"],
            ascending=[False, False]
        )
        .head(top_k)
        .reset_index()
    )

    return aggregated


In [20]:
def extract_top_tfidf_features_for_mix_model(
    model,
    vectorizer,
    num_features_count,
    top_k=10
):
    feature_names = vectorizer.get_feature_names_out()

    # FULL coefficient vector
    coefs = model.coef_[0]

    # Slice only TF-IDF part
    tfidf_feature_count = len(vectorizer.get_feature_names_out())

    tfidf_coefs = coefs[
    num_features_count : num_features_count + tfidf_feature_count
    ]
    
    # Safety check
    assert len(tfidf_coefs) == len(feature_names)

    top_pos_idx = tfidf_coefs.argsort()[-top_k:][::-1]
    top_neg_idx = tfidf_coefs.argsort()[:top_k]

    label_1 = pd.DataFrame({
        "feature": feature_names[top_pos_idx],
        "coefficient": tfidf_coefs[top_pos_idx]
    })

    label_0 = pd.DataFrame({
        "feature": feature_names[top_neg_idx],
        "coefficient": tfidf_coefs[top_neg_idx]
    })

    return {
        "label_1": label_1,
        "label_0": label_0
    }

def extract_top_mixed_features(
    model,
    vectorizer,
    num_feature_names,
    embedding_dim=384,
    top_k=10
):

    tfidf_feature_names = vectorizer.get_feature_names_out()

    embedding_feature_names = np.array(
        [f"embed_{i}" for i in range(embedding_dim)]
    )

    all_feature_names = np.concatenate([
        num_feature_names,
        tfidf_feature_names,
        embedding_feature_names
    ])

    coefs = model.coef_[0]

    assert len(coefs) == len(all_feature_names)

    top_pos_idx = coefs.argsort()[-top_k:][::-1]
    top_neg_idx = coefs.argsort()[:top_k]

    def get_type(i):
        if i < len(num_feature_names):
            return "numerical"
        elif i < len(num_feature_names) + len(tfidf_feature_names):
            return "tfidf"
        else:
            return "embedding"

    label_1 = pd.DataFrame({
        "feature": all_feature_names[top_pos_idx],
        "coefficient": coefs[top_pos_idx],
        "feature_type": [get_type(i) for i in top_pos_idx]
    })

    label_0 = pd.DataFrame({
        "feature": all_feature_names[top_neg_idx],
        "coefficient": coefs[top_neg_idx],
        "feature_type": [get_type(i) for i in top_neg_idx]
    })

    return {
        "label_1": label_1,
        "label_0": label_0
    }



In [26]:
import numpy as np
from scipy.sparse import hstack
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sentence_transformers import SentenceTransformer
from scipy.sparse import csr_matrix


def run_numerical_tfidf_embed_model(
    df_train,
    df_test,
    train_embeddings,
    test_embeddings,
    label_col,
    text_col="text_for_feature",
    num_cols=None,
    scaler=None,
    vectorizer=None,
    model=None,
    model_name="numerical_tfidf",
    fold_number=None,
    test_idx=None
):
    """
    Trains and evaluates a Numerical + TF-IDF + Sentence Embedding model.
    """

    if num_cols is None:
        num_cols = df_train.select_dtypes(include="number").columns.tolist()
        num_cols.remove(label_col)

    if scaler is None:
        scaler = StandardScaler()

    if vectorizer is None:
        vectorizer = TfidfVectorizer(
            ngram_range=(1, 1),
            min_df=5,
            max_features=3000,
            stop_words="english"
        )


    if model is None:
        model = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1
    )

    # -------------------------
    # Numerical features
    # -------------------------
    X_train_num = scaler.fit_transform(df_train[num_cols])
    X_test_num = scaler.transform(df_test[num_cols])

    # -------------------------
    # TF-IDF features
    # -------------------------
    X_train_text = vectorizer.fit_transform(df_train[text_col])
    X_test_text = vectorizer.transform(df_test[text_col])

    # -------------------------
    # Sentence embeddings
    # -------------------------

    ## Scale the embedding
    embed_scaler = StandardScaler()
    X_train_embed = embed_scaler.fit_transform(train_embeddings)
    X_test_embed = embed_scaler.transform(test_embeddings)

    # Convert to sparse for stacking
    X_train_embed = csr_matrix(X_train_embed)
    X_test_embed = csr_matrix(X_test_embed)

    # -------------------------
    # Concatenate ALL features
    # -------------------------
    X_train = hstack([
        X_train_num,
        X_train_text,
        X_train_embed
    ])

    X_test = hstack([
        X_test_num,
        X_test_text,
        X_test_embed
    ])

    y_train = df_train[label_col]
    y_test = df_test[label_col]

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    confidence = np.max(y_proba, axis=1)
    prob_label1 = y_proba[:, 1]

    report = classification_report(y_test, y_pred)

    errors_df = build_error_df(
        df_test=df_test,
        y_true=y_test,
        y_pred=y_pred,
        model_name=model_name,
        fold_number=fold_number,
        test_idx=test_idx
    )

    f1 = f1_score(y_test, y_pred)

    errors_df["error_type"] = ""
    errors_df["error_subtype"] = ""
    errors_df["notes"] = ""
    errors_df["confidence"] = confidence
    errors_df["prob_label1"] = prob_label1

    return {
        "model": model,
        "scaler": scaler,
        "vectorizer": vectorizer,
        "classification_report": report,
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_score": f1,
        "errors_df": errors_df,
        "y_pred": y_pred
    }

In [33]:
from sklearn.preprocessing import normalize

import pandas as pd
import numpy as np


def run_k_fold_numerical_tfidf_error_analysis(
    df: pd.DataFrame,
    folds,
    n_splits=5,
    label_col="effort",
    text_col="text_for_feature",
    num_cols=None,
    model_name="numerical_tfidf",
    top_k_features=10
):
    """
    Runs k-fold cross-validation error analysis for a mixed
    Numerical + TF-IDF linear model.
    """

    if num_cols is None:
        num_cols = []

    # Load embedder once
    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    # Compute embeddings once for full dataset
    all_embeddings = embedder.encode(
        df[text_col].tolist(),
        show_progress_bar=True
    )

    all_embeddings = normalize(all_embeddings)

    all_errors = []
    fold_tfidf_features = []
    fold_mixed_features = []
    fold_accuracies = []
    fold_f1_scores = []

    for fold_info in folds:

        fold = fold_info["fold"]
        train_idx = fold_info["train_idx"]
        test_idx = fold_info["test_idx"]

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        train_embed = all_embeddings[train_idx]
        test_embed = all_embeddings[test_idx]

        result = run_numerical_tfidf_embed_model(
            df_train=train_df,
            df_test=test_df,
            train_embeddings=train_embed,
            test_embeddings=test_embed,
            label_col=label_col,
            text_col=text_col,
            num_cols=num_cols,
            model_name=f"{model_name}_fold_{fold}",
            fold_number=fold,
            test_idx=test_idx
        )

        fold_accuracies.append(result["accuracy"])
        fold_f1_scores.append(result["f1_score"])

        # errors already contain fold metadata
        all_errors.append(result["errors_df"])

        

        # ----- TF-IDF features -----
        tfidf_feats = extract_top_tfidf_features_for_mix_model(
            model=result["model"],
            vectorizer=result["vectorizer"],
            num_features_count=len(num_cols),
            top_k=top_k_features
        )

        fold_tfidf_features.append({
            "label_1_features": tfidf_feats["label_1"],
            "label_0_features": tfidf_feats["label_0"]
        })

        # ----- Mixed features -----
        mixed_feats = extract_top_mixed_features(
            model=result["model"],
            vectorizer=result["vectorizer"],
            num_feature_names=np.array(num_cols),
            top_k=top_k_features
        )

        fold_mixed_features.append({
            "label_1_features": mixed_feats["label_1"],
            "label_0_features": mixed_feats["label_0"]
        })

    # =============================
    # Metrics aggregation
    # =============================

    mean_accuracy = np.mean(fold_accuracies)
    mean_f1_score = np.mean(fold_f1_scores)

    # =============================
    # Error aggregation
    # =============================

    errors_all = pd.concat(all_errors).reset_index(drop=True)
    errors_only = errors_all[errors_all["correct"] == False]

    errors_all["error_type"] = "N/A"
    errors_all["error_subtype"] = "N/A"
    errors_all["notes"] = ""

    true_label_counts = errors_only["true_label"].value_counts()
    pred_label_counts = errors_only["pred_label"].value_counts()
    error_group_counts = (
        errors_only
        .groupby(["true_label", "pred_label"])
        .size()
    )

    # =============================
    # Feature aggregation
    # =============================

    stable_tfidf_label_1 = aggregate_fold_features(
        fold_tfidf_features,
        label_key="label_1_features",
        top_k=top_k_features
    )

    stable_tfidf_label_0 = aggregate_fold_features(
        fold_tfidf_features,
        label_key="label_0_features",
        top_k=top_k_features
    )

    stable_mixed_label_1 = aggregate_fold_features(
        fold_mixed_features,
        label_key="label_1_features",
        top_k=top_k_features
    )

    stable_mixed_label_0 = aggregate_fold_features(
        fold_mixed_features,
        label_key="label_0_features",
        top_k=top_k_features
    )

    common_mixed_features = (
        set(stable_mixed_label_1["feature"]) &
        set(stable_mixed_label_0["feature"])
    )

    common_mixed_df = (
        stable_mixed_label_1
        .merge(
            stable_mixed_label_0,
            on="feature",
            suffixes=("_label_1", "_label_0")
        )
        .assign(is_common_feature=True)
    )

    return {
        "errors_all": errors_all,
        "errors_only": errors_only,
        "true_label_counts": true_label_counts,
        "pred_label_counts": pred_label_counts,
        "error_group_counts": error_group_counts,
        "fold_metrics": {
            "accuracy_per_fold": fold_accuracies,
            "f1_per_fold": fold_f1_scores,
            "mean_accuracy": mean_accuracy,
            "mean_f1_score": mean_f1_score
        },
        "stable_tfidf_features": {
            "label_1": stable_tfidf_label_1,
            "label_0": stable_tfidf_label_0
        },
        "stable_mixed_features": {
            "label_1": stable_mixed_label_1,
            "label_0": stable_mixed_label_0
        },
        "common_mixed_features": {
            "feature_set": common_mixed_features,
            "feature_df": common_mixed_df
        }
    }

In [34]:
df = pd.read_csv("../../../data/post_quality/processed/effort_final_interpretable_features_dataset.csv")
df.tail(5)

,Unnamed: 0,title,body,effort,text_for_feature,num_paragraphs,has_multi_paragraphs,text,avg_sentence_length,num_tokens,...,question_count,has_contextual_grounding,has_temporal_progression,informational_question,opinion_with_experience,opinion_experience_long,personal_problem_question,rant_vent_storytelling,self_reflection_detector,curiosity_detector
235,235,Does being an introvert actually lead to highe...,There’s a common belief that introverts tend t...,1,Does being an introvert actually lead to highe...,4,1,Does being an introvert actually lead to highe...,15.833333,95,...,3,1,0,False,False,False,False,False,False,True
236,236,Does anything seem legendary anymore?,I was having a conversation with a friend as t...,1,Does anything seem legendary anymore? I was ha...,5,1,Does anything seem legendary anymore? I was ha...,14.062500,225,...,3,1,1,False,True,True,False,True,True,True
237,237,How will US government deal with a large propo...,How will US government deal with a large propo...,1,How will US government deal with a large propo...,3,1,How will US government deal with a large propo...,14.333333,129,...,6,1,0,True,False,False,False,False,False,True
238,238,Why so many straight men are constantly polici...,I’ve noticed a pattern where straight guys lov...,1,Why so many straight men are constantly polici...,2,1,Why so many straight men are constantly polici...,13.857143,97,...,4,1,0,False,False,False,False,False,True,False
239,239,Are we done?,Imagine the year is 2050 AI has evolved into A...,1,Are we done? Imagine the year is 2050 AI has e...,3,1,Are we done? Imagine the year is 2050 AI has e...,20.800000,104,...,4,1,0,True,False,False,False,True,True,True


In [35]:
for col in df.columns:
    print(f"Number of null value in '{col}' column is {df[col].isnull().sum()}")

Number of null value in 'Unnamed: 0' column is 0
Number of null value in 'title' column is 0
Number of null value in 'body' column is 25
Number of null value in 'effort' column is 0
Number of null value in 'text_for_feature' column is 0
Number of null value in 'num_paragraphs' column is 0
Number of null value in 'has_multi_paragraphs' column is 0
Number of null value in 'text' column is 0
Number of null value in 'avg_sentence_length' column is 0
Number of null value in 'num_tokens' column is 0
Number of null value in 'has_first_person' column is 0
Number of null value in 'has_attempted_marker' column is 0
Number of null value in 'has_constraint_marker' column is 0
Number of null value in 'question_count' column is 0
Number of null value in 'has_contextual_grounding' column is 0
Number of null value in 'has_temporal_progression' column is 0
Number of null value in 'informational_question' column is 0
Number of null value in 'opinion_with_experience' column is 0
Number of null value in '

In [36]:
df["body"] = df["body"].fillna("")
for col in df.columns:
    print(f"Number of null value in '{col}' column is {df[col].isnull().sum()}")

Number of null value in 'Unnamed: 0' column is 0
Number of null value in 'title' column is 0
Number of null value in 'body' column is 0
Number of null value in 'effort' column is 0
Number of null value in 'text_for_feature' column is 0
Number of null value in 'num_paragraphs' column is 0
Number of null value in 'has_multi_paragraphs' column is 0
Number of null value in 'text' column is 0
Number of null value in 'avg_sentence_length' column is 0
Number of null value in 'num_tokens' column is 0
Number of null value in 'has_first_person' column is 0
Number of null value in 'has_attempted_marker' column is 0
Number of null value in 'has_constraint_marker' column is 0
Number of null value in 'question_count' column is 0
Number of null value in 'has_contextual_grounding' column is 0
Number of null value in 'has_temporal_progression' column is 0
Number of null value in 'informational_question' column is 0
Number of null value in 'opinion_with_experience' column is 0
Number of null value in 'o

In [37]:
df.head()

,Unnamed: 0,title,body,effort,text_for_feature,num_paragraphs,has_multi_paragraphs,text,avg_sentence_length,num_tokens,...,question_count,has_contextual_grounding,has_temporal_progression,informational_question,opinion_with_experience,opinion_experience_long,personal_problem_question,rant_vent_storytelling,self_reflection_detector,curiosity_detector
0,0,If your first-ever attempt at gambling went co...,,0,If your first-ever attempt at gambling went co...,0,0,If your first-ever attempt at gambling went co...,8.0,15,...,0,0,1,False,False,False,False,False,False,False
1,1,"According to birds, all land animals are botto...",,0,"According to birds, all land animals are botto...",0,0,"According to birds, all land animals are botto...",9.0,9,...,0,0,0,False,False,False,False,False,False,False
2,2,Very few people can actually keep their watch ...,,0,Very few people can actually keep their watch ...,0,0,Very few people can actually keep their watch ...,13.0,13,...,0,0,0,False,False,False,False,False,False,False
3,3,"What is keeping the really deadly diseases, li...",,0,"What is keeping the really deadly diseases, li...",0,0,"What is keeping the really deadly diseases, li...",15.0,15,...,1,0,0,True,False,False,False,False,False,False
4,4,When did humans start living indoors?,,0,When did humans start living indoors?,0,0,When did humans start living indoors?,6.0,6,...,1,0,0,True,False,False,False,False,False,False


In [42]:
df.drop(columns=['Unnamed: 0'],axis=1,inplace=True)

## Experimenting tf-idf + numerical + sentence transformer model

In [43]:
num_features_v1 = df.select_dtypes(exclude=['object']).columns.tolist()
num_features_v1.pop(0)
num_features_v1

['num_paragraphs',
 'has_multi_paragraphs',
 'avg_sentence_length',
 'num_tokens',
 'has_first_person',
 'has_attempted_marker',
 'has_constraint_marker',
 'question_count',
 'has_contextual_grounding',
 'has_temporal_progression',
 'informational_question',
 'opinion_with_experience',
 'opinion_experience_long',
 'personal_problem_question',
 'rant_vent_storytelling',
 'self_reflection_detector',
 'curiosity_detector']

In [44]:
folds = generate_stratified_folds(df=df)
result_v1 = run_k_fold_numerical_tfidf_error_analysis(df=df,folds=folds,num_cols=num_features_v1,model_name="numerical_tfidf_embed")

Batches: 100%|██████████| 8/8 [00:03<00:00,  2.45it/s]
c:\Users\prakh\projects\discusso-ml\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\prakh\projects\discusso-ml\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\prakh\projects\discusso-ml\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\prakh\projects\discusso-ml\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_

In [45]:
result_v1["stable_mixed_features"]["label_0"]

,feature,freq,mean_coef,mean_abs_coef
0,embed_319,5,-0.418928,0.418928
1,embed_42,5,-0.334592,0.334592
2,embed_128,5,-0.325670,0.325670
3,embed_121,3,-0.317869,0.317869
4,embed_48,3,-0.305666,0.305666
5,embed_0,3,-0.269568,0.269568
6,embed_69,2,-0.383283,0.383283
7,informational_question,2,-0.319061,0.319061
8,embed_47,2,-0.318044,0.318044
9,embed_40,2,-0.309832,0.309832


In [46]:
result_v1["stable_mixed_features"]["label_1"]

,feature,freq,mean_coef,mean_abs_coef
0,question_count,5,0.625367,0.625367
1,has_multi_paragraphs,5,0.373551,0.373551
2,curiosity_detector,5,0.369157,0.369157
3,embed_141,5,0.341410,0.341410
4,embed_239,5,0.310464,0.310464
5,has_first_person,4,0.300606,0.300606
6,embed_74,3,0.338282,0.338282
7,embed_166,3,0.333267,0.333267
8,embed_275,2,0.306231,0.306231
9,embed_170,2,0.293277,0.293277


In [47]:
result_v1["stable_tfidf_features"]["label_0"]

,feature,freq,mean_coef,mean_abs_coef
0,like,4,-0.039048,0.039048
1,country,4,-0.025664,0.025664
2,wouldn,4,-0.025169,0.025169
3,talking,3,-0.032389,0.032389
4,really,3,-0.031865,0.031865
5,did,3,-0.031208,0.031208
6,bad,3,-0.027269,0.027269
7,parents,2,-0.025571,0.025571
8,things,2,-0.024640,0.024640
9,young,2,-0.024638,0.024638


In [48]:
result_v1["stable_tfidf_features"]["label_1"]

,feature,freq,mean_coef,mean_abs_coef
0,appreciated,5,0.020584,0.020584
1,wall,4,0.031253,0.031253
2,mean,4,0.024868,0.024868
3,social,4,0.021185,0.021185
4,appreciate,3,0.020744,0.020744
5,large,3,0.020423,0.020423
6,access,3,0.019660,0.019660
7,use,2,0.025059,0.025059
8,working,2,0.024423,0.024423
9,question,2,0.021209,0.021209


In [49]:
errors_v1 = result_v1["errors_only"]
errors_v1.shape

(47, 34)

In [54]:
errors_v1

,title,body,effort,text_for_feature,num_paragraphs,has_multi_paragraphs,text,avg_sentence_length,num_tokens,has_first_person,...,true_label,pred_label,correct,model,fold,error_type,error_subtype,notes,confidence,prob_label1
8,The snyderverse subreddit is like a cult.,Some of the rules saying you can’t disrespect ...,0,The snyderverse subreddit is like a cult. Some...,1,0,The snyderverse subreddit is like a cult. Some...,29.333333,88,1,...,0,1,False,numerical_tfidf_embed_fold_0,0,,,,0.623691,0.623691
11,I spent all week looking for tax credits/deduc...,My business partner was finalizing the 2025 nu...,0,I spent all week looking for tax credits/deduc...,1,0,I spent all week looking for tax credits/deduc...,38.000000,38,1,...,0,1,False,numerical_tfidf_embed_fold_0,0,,,,0.522044,0.522044
13,Urgent question on London flight connection,I have found a aer lingus flight from Dublin t...,1,Urgent question on London flight connection I ...,1,0,Urgent question on London flight connection I ...,12.833333,75,1,...,1,0,False,numerical_tfidf_embed_fold_0,0,,,,0.986419,0.013581
23,When and why did athletes in team sports start...,As the question states; curious about why this...,0,When and why did athletes in team sports start...,1,0,When and why did athletes in team sports start...,11.200000,56,0,...,0,1,False,numerical_tfidf_embed_fold_0,0,,,,0.908612,0.908612
30,"Would You Rather: Get 25,000 USD or receive no...","""Traffic related violations"" means any law rel...",0,"Would You Rather: Get 25,000 USD or receive no...",2,1,"Would You Rather: Get 25,000 USD or receive no...",15.750000,63,0,...,0,1,False,numerical_tfidf_embed_fold_0,0,,,,0.702879,0.702879
56,i physically cannot see other humans as the sa...,"TW for rape, csa, incest and cult mentions but...",0,i physically cannot see other humans as the sa...,7,1,i physically cannot see other humans as the sa...,34.062500,545,1,...,0,1,False,numerical_tfidf_embed_fold_1,1,,,,0.598211,0.598211
59,Gym Locker Room,I belong to the local gym where everybody is r...,0,Gym Locker Room I belong to the local gym wher...,3,1,Gym Locker Room I belong to the local gym wher...,15.434783,355,1,...,0,1,False,numerical_tfidf_embed_fold_1,1,,,,0.841262,0.841262
62,Got kicked out without a backup plan,"Hello, I’m 22F and currently living with my pa...",0,"Got kicked out without a backup plan Hello, I’...",5,1,"Got kicked out without a backup plan Hello, I’...",21.090909,232,1,...,0,1,False,numerical_tfidf_embed_fold_1,1,,,,0.982650,0.982650
64,I'm forcing my boss to arrive at the office ev...,I just started a new job and decided I want to...,0,I'm forcing my boss to arrive at the office ev...,6,1,I'm forcing my boss to arrive at the office ev...,18.733333,280,1,...,0,1,False,numerical_tfidf_embed_fold_1,1,,,,0.993904,0.993904
65,Why is there mold on my ceiling?,"Hi, having lived in the same house for 15 year...",1,"Why is there mold on my ceiling? Hi, having li...",1,0,"Why is there mold on my ceiling? Hi, having li...",13.833333,83,1,...,1,0,False,numerical_tfidf_embed_fold_1,1,,,,0.608759,0.391241


In [51]:
fp_v1 = errors_v1[(errors_v1["pred_label"] == 1) & (errors_v1["true_label"]==0)]
fp_v1.shape

(24, 34)

In [52]:
fn_v1 = errors_v1[(errors_v1["pred_label"] == 0) & (errors_v1["true_label"]==1)]
fn_v1.shape

(23, 34)

## Model Development Summary

We developed a hybrid interpretable model for effort classification using:

• Structural features (post length, paragraph structure)
• Behavioral markers (first-person usage, attempt markers)
• TF-IDF lexical features
• Sentence embeddings (MiniLM)

The modeling process followed an iterative error-driven approach:

1. Start with interpretable numerical and lexical features
2. Perform detailed error analysis
3. Add new features only when a systematic error pattern appeared
4. Re-evaluate model performance
5. Stop feature development once no new error buckets were discovered

This approach prioritizes interpretability and avoids unnecessary model complexity.

## Effect of Semantic Embeddings

To test whether implicit semantic relationships contributed to classification errors,
we extended the model with sentence embeddings using the MiniLM sentence transformer.

Model comparison:

Numerical + TF-IDF:
Errors = 52

Numerical + TF-IDF + Sentence Embeddings:
Errors = 47

The addition of embeddings reduced errors slightly (≈5 posts) but did not significantly
change overall performance.

This suggests that most predictive signal is already captured by lexical and
behavioral features rather than deeper semantic similarity.

## Error Pattern Analysis

Manual inspection of remaining errors revealed two recurring patterns.

### 1. Personal Narrative / Rant Posts

Approximately 25% of errors occur in posts that contain personal stories
or emotional narratives. These posts often mix reflective language with
complaint or venting, making them difficult to categorize consistently.

Example pattern:
• personal experiences
• emotional tone
• implicit questions rather than explicit requests

These posts blur the boundary between effortful reflection and low-effort venting.

---

### 2. Short Question Posts

Another portion of errors occurs in very short question posts.

Example pattern:
"Why do people ghost?"

Short questions often lack enough contextual signals to determine whether
the author is seeking genuine understanding or expressing rhetorical frustration.

This ambiguity limits reliable classification.

## Interpretation

The remaining errors appear to stem primarily from ambiguity in the underlying
content rather than missing model features.

Two factors contribute to this:

1. Ambiguous intent in personal narrative posts
2. Insufficient context in short question posts

Because these issues arise from the inherent structure of the data rather
than feature limitations, further model complexity is unlikely to produce
substantial improvements.

Therefore, the current model is considered to have reached a practical
performance ceiling given the available signals.

## Final Model

The final model combines:

* Structural features
* Behavioral indicators
* TF-IDF lexical features
* Sentence transformer embeddings

Classification is performed using a logistic regression classifier with
balanced class weighting.

This architecture maintains interpretability while incorporating limited
semantic representation.

## Future Work

Future improvements could explore:

* Better annotation guidelines to reduce label ambiguity
* Context-aware modeling of personal narrative posts
* More detailed discourse features
* Larger datasets to improve generalization

However, given the current dataset, the largest remaining source of
error appears to be inherent ambiguity in the text rather than
missing model features.